# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [ ]:
#!pip install langchain_community langchain_experimental langchain_openai
"""!uv pip install -U \
  "langchain==0.2.16" \
  "langchain-core==0.2.38" \
  "langchain_community==0.2.16" \
  "langchain_openai==0.1.23" \
  "langchain_experimental" \
  "langchain_text_splitters==0.2.4" \
  "chromadb==0.5.5" \
  "openai==1.40.0" \
  "httpx==0.27.2" \
  "beautifulsoup4==4.12.3""""

Resolved 115 packages in 900ms                                       
⠹ Preparing packages... (0/1)                                                   
⠹ Preparing packages... (0/1)---------------------     0 B/202.30 KiB        
⠹ Preparing packages... (0/1)--------------------- 16.00 KiB/202.30 KiB      
⠹ Preparing packages... (0/1)--------------------- 32.00 KiB/202.30 KiB      
⠹ Preparing packages... (0/1)--------------------- 48.00 KiB/202.30 KiB      
⠹ Preparing packages... (0/1)--------------------- 64.00 KiB/202.30 KiB      
⠹ Preparing packages... (0/1)2m------------------- 80.00 KiB/202.30 KiB      
⠹ Preparing packages... (0/1)m---------------- 96.00 KiB/202.30 KiB      
⠹ Preparing packages... (0/1)30m-------------- 112.00 KiB/202.30 KiB     
⠹ Preparing packages... (0/1)------------ 128.00 KiB/202.30 KiB     
⠹ Preparing packages... (0/1)------------ 144.00 KiB/202.30 KiB     
⠹ Preparing packages... (0/1)------------ 160.00 KiB/202.30 KiB     
⠹ Preparing packages... (0

In [12]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [ ]:
# !uv pip install faiss-cpu > /dev/null
# !uv pip install google-search-results > /dev/null
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

Resolved 3 packages in 253ms                                         
⠙ Preparing packages... (0/1)                                                   
⠙ Preparing packages... (0/1)-------------------     0 B/6.60 MiB            
⠙ Preparing packages... (0/1)------------------- 16.00 KiB/6.60 MiB          
⠙ Preparing packages... (0/1)------------------- 32.00 KiB/6.60 MiB          
⠙ Preparing packages... (0/1)------------------- 48.00 KiB/6.60 MiB          
⠙ Preparing packages... (0/1)------------------- 62.26 KiB/6.60 MiB          
⠙ Preparing packages... (0/1)------------------- 78.26 KiB/6.60 MiB          
⠙ Preparing packages... (0/1)------------------- 94.26 KiB/6.60 MiB          
⠙ Preparing packages... (0/1)------------------- 110.26 KiB/6.60 MiB         
⠙ Preparing packages... (0/1)------------------- 126.26 KiB/6.60 MiB         
⠙ Preparing packages... (0/1)------------------- 142.26 KiB/6.60 MiB         
⠙ Preparing packages... (0/1)------------------- 158.26 KiB/6.60 MiB 

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [20]:
# Define your embedding model
embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [21]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

/var/folders/zh/vbl6k4hs1ggfwjfqn6t_gb100000gn/T/ipykernel_6685/842275654.py:9: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)


In [22]:
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/var/folders/zh/vbl6k4hs1ggfwjfqn6t_gb100000gn/T/ipykernel_6685/3064747660.py:4: LangChainDeprecationWarning: The class `ZeroShotAgent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use create_react_agent instead.
  agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [23]:
OBJECTIVE = "Write a weather report for SF today"

In [24]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [25]:
baby_agi({"objective": OBJECTIVE})

/var/folders/zh/vbl6k4hs1ggfwjfqn6t_gb100000gn/T/ipykernel_6685/3867971396.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.
  baby_agi({"objective": OBJECTIVE})



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should create a todo list for the objective
Action: TODO
Action Input: Write a weather report for SF today
Observation: 

1. Check the current weather conditions in San Francisco.
2. Gather information on the temperature, humidity, wind speed, and precipitation.
3. Research any potential weather patterns or systems that may affect the current conditions.
4. Look at historical data and trends for this time of year in San Francisco.
5. Consult with local meteorologists or weather experts for their insights and predictions.
6. Create a rough outline or structure for the weather report, including sections for current conditions, forecast, and any notable weather events.
7. Begin writing the report, starting with a brief overview of the current conditions.
8. Use descriptive language and visuals (such as maps or charts) to enhance the report.
9. Include any 

{'objective': 'Write a weather report for SF today'}

# Exercise

## Change the obejective

In [32]:
# Objective
OBJECTIVE = "Plan a 3 day budget-friendly trip from Madrid, Spain to Lisbon, Portugal for under 1000€."

# New Prompt
budget_prompt = PromptTemplate.from_template(
    "You are a travel budget expert. Given the following items and estimated costs, "
    "add them up and tell me if they fit under the stated budget, and suggest cuts if not.\n"
    "Input: {input}"
)

# Model
budget_chain = LLMChain(llm=OpenAI(temperature=0), prompt=budget_prompt)

# Build a NEW tools list from scratch — don't extend the old global `tools`
search = SerpAPIWrapper()
todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. "
    "Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)

tools_trip = [
    Tool(name="Search", func=search.run,
         description="useful for finding current info on flights, hotels, attractions, and prices"),
    Tool(name="TODO", func=todo_chain.run,
         description="useful for creating a todo list for a travel-planning objective"),
    Tool(name="BudgetCheck", func=budget_chain.run,
         description="useful for checking whether a list of estimated costs fits within a budget. "
                     "Input: a description of costs and the budget limit."),
]

# New Prompt
prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools_trip,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

# Rebuild the agent
llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools_trip]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(agent=agent, tools=tools_trip, verbose=True)

# Fresh vectorstore for this independent run
embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore_trip = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

# Run BabyAGI with the new execution chain
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore_trip,
    task_execution_chain=agent_executor,
    verbose=False,
    max_iterations=3,
)

baby_agi({"objective": OBJECTIVE})

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should use the TODO function to create a list of tasks
Action: TODO
Action Input: "Create a todo list for a 3 day trip from Madrid, Spain to Lisbon, Portugal"
Observation: 

1. Research and book transportation: Look into flights, trains, or buses from Madrid to Lisbon and book the most convenient and cost-effective option.

2. Plan accommodations: Research and book a hotel or Airbnb in Lisbon for the duration of the trip. Consider location, amenities, and budget.

3. Create a budget: Determine how much you are willing to spend on the trip and allocate funds for transportation, accommodations, food, and activities.

4. Make a list of must-see attractions: Research and make a list of the top attractions in Lisbon that you want to visit during your trip.

5. Plan daily itineraries: Based on your list of must-see attractions, create a daily itinerary for ea

{'objective': 'Plan a 3 day budget-friendly trip from Madrid, Spain to Lisbon, Portugal for under 1000€.'}